In [2]:
import numpy as np
from sklearn.neighbors import KernelDensity
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from collections import defaultdict
import matplotlib.pyplot as plt
import pandas as pd


In [5]:
data = pd.read_csv('overview_data.csv')

In [22]:
data

,Unnamed: 0,overview_matchesplayed,overview_matcheswon,overview_matcheslost,overview_matchesabandoned,overview_timeplayed,overview_kills,overview_deaths,overview_attackerroundswon,overview_attackerteamkillsinobj,...,overview_playstyledefenderintelprovider,overview_playstyledefendersupporter,overview_playstyledefendertrapper,overview_playstyledefenderutilitydenier,overview_kdratio,overview_killspermatch,overview_killspermin,overview_winpercentage,overview_timealive,is_cheater
0,0,37,31,6,0,0,330,66,61,0,...,0,0,0,0,5.00,8.92,330.00,83.78,0,1
1,1,148,73,74,1,212365,550,515,1998,0,...,42,32,23,38,1.07,3.72,0.16,49.32,412,1
2,2,121,73,21,0,0,849,274,158,0,...,0,0,0,0,3.10,7.02,849.00,60.33,0,1
3,3,99,64,35,0,0,767,398,179,0,...,240,149,2,109,1.93,7.75,767.00,64.65,0,1
4,4,79,79,0,0,0,961,21,169,0,...,0,0,0,0,45.76,12.16,961.00,100.00,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3199,3199,324,237,67,20,1912127,1922,1004,16165,5,...,4095,5201,2495,5962,1.91,5.93,0.06,73.15,1904,0
3200,3200,384,198,14,172,322896,224,88,467,0,...,47,326,278,285,2.55,0.58,0.04,51.56,3669,0
3201,3201,398,252,138,8,9833873,2101,1521,12134,7,...,7153,7091,5967,7977,1.38,5.28,0.01,63.32,6465,0
3202,3202,946,492,440,14,1382718,6379,4264,5017,1,...,1213,538,669,436,1.50,6.74,0.28,52.01,324,0


In [23]:
X = data.drop(columns=['Unnamed: 0', 'is_cheater'])

In [32]:
print(X.columns)

Index(['overview_matchesplayed', 'overview_matcheswon', 'overview_matcheslost',
       'overview_matchesabandoned', 'overview_timeplayed', 'overview_kills',
       'overview_deaths', 'overview_attackerroundswon',
       'overview_attackerteamkillsinobj', 'overview_attackerenemykillsinobj',
       'overview_attackerteamkillsoutobj', 'overview_attackerenemykillsoutobj',
       'overview_defenderroundswon', 'overview_defenderteamkillsinobj',
       'overview_defenderenemykillsinobj', 'overview_defenderteamkillsoutobj',
       'overview_defenderenemykillsoutobj', 'overview_headshots',
       'overview_headshotsmissed', 'overview_headshotpercentage',
       'overview_wallbangs', 'overview_damagedealt', 'overview_assists',
       'overview_teamkills', 'overview_playstyleattackerbreacher',
       'overview_playstyleattackerentryfragger',
       'overview_playstyleattackerintelprovider',
       'overview_playstyleattackerroamclearer',
       'overview_playstyleattackersupporter',
       'overv

In [24]:
y = data['is_cheater']

In [25]:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

In [26]:
class KDENaiveBayes:
    def __init__(self, bandwidth=1.0, kernel='gaussian'):
        self.bandwidth = bandwidth
        self.kernel = kernel
        self.kde_models = defaultdict(dict)
        self.class_priors = {}
        self.classes_ = None

    def fit(self, X, y):
        self.classes_ = np.unique(y)

        for cls in self.classes_:
            X_cls = X[y == cls]
            self.class_priors[cls] = len(X_cls) / len(y)

            for feature_index in range(X.shape[1]):
                feature_vals = X_cls[:, feature_index]

                # Remove noisy zeros if they make up more than 10%
                zero_mask = feature_vals == 0
                zero_fraction = np.mean(zero_mask)

                if zero_fraction > 0.10:
                    feature_vals = feature_vals[~zero_mask]  # remove zeros

                kde = KernelDensity(kernel=self.kernel, bandwidth=self.bandwidth)
                kde.fit(feature_vals.reshape(-1, 1))
                self.kde_models[cls][feature_index] = kde

    def predict_log_proba(self, X):
        log_probs = np.zeros((X.shape[0], len(self.classes_)))

        for i, cls in enumerate(self.classes_):
            log_prior = np.log(self.class_priors[cls])
            total_log_likelihood = np.zeros(X.shape[0])

            for feature_index in range(X.shape[1]):
                kde = self.kde_models[cls][feature_index]
                feature_vals = X[:, feature_index].reshape(-1, 1)
                log_dens = kde.score_samples(feature_vals)
                total_log_likelihood += log_dens

            log_probs[:, i] = log_prior + total_log_likelihood

        return log_probs

    def predict(self, X):
        log_probs = self.predict_log_proba(X)
        return self.classes_[np.argmax(log_probs, axis=1)]


In [27]:
model = KDENaiveBayes(bandwidth=2.0)  # Tune this bandwidth
model.fit(X_train.values, y_train.values)

y_pred = model.predict(X_test.values)
accuracy = np.mean(y_pred == y_test.values)

print(f"Accuracy: {accuracy:.3f}")


Accuracy: 0.866


In [28]:
from sklearn.model_selection import KFold
import numpy as np

def cross_validate_bandwidth(X, y, bandwidths, n_splits=5):
    X = X.values if hasattr(X, "values") else X
    y = y.values if hasattr(y, "values") else y

    best_bandwidth = None
    best_score = -np.inf
    scores_per_bandwidth = {}

    for bw in bandwidths:
        print(f"Testing bandwidth: {bw}")
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
        fold_scores = []

        for train_index, val_index in kf.split(X):
            X_train, X_val = X[train_index], X[val_index]
            y_train, y_val = y[train_index], y[val_index]

            model = KDENaiveBayes(bandwidth=bw)
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
            accuracy = np.mean(y_pred == y_val)
            fold_scores.append(accuracy)

        mean_score = np.mean(fold_scores)
        scores_per_bandwidth[bw] = mean_score
        print(f"Bandwidth: {bw}, Mean CV Accuracy: {mean_score:.4f}")

        if mean_score > best_score:
            best_score = mean_score
            best_bandwidth = bw

    print(f"\n✅ Best bandwidth: {best_bandwidth} (CV Accuracy: {best_score:.4f})")
    return best_bandwidth, scores_per_bandwidth


In [29]:
bandwidth_range = np.linspace(0.1, 5.0, 10)  # Try 10 values from 0.1 to 5.0
best_bw, all_scores = cross_validate_bandwidth(X_train, y_train, bandwidth_range)

# Fit model with best bandwidth and evaluate on test set
model = KDENaiveBayes(bandwidth=best_bw)
model.fit(X_train.values, y_train.values)
y_pred = model.predict(X_test.values)
test_accuracy = np.mean(y_pred == y_test.values)

print(f"\n🎯 Final Test Accuracy with best bandwidth ({best_bw}): {test_accuracy:.4f}")


Testing bandwidth: 0.1
Bandwidth: 0.1, Mean CV Accuracy: 0.8502
Testing bandwidth: 0.6444444444444445
Bandwidth: 0.6444444444444445, Mean CV Accuracy: 0.8502
Testing bandwidth: 1.188888888888889
Bandwidth: 1.188888888888889, Mean CV Accuracy: 0.8502
Testing bandwidth: 1.7333333333333336
Bandwidth: 1.7333333333333336, Mean CV Accuracy: 0.8502
Testing bandwidth: 2.277777777777778
Bandwidth: 2.277777777777778, Mean CV Accuracy: 0.8502
Testing bandwidth: 2.8222222222222224
Bandwidth: 2.8222222222222224, Mean CV Accuracy: 0.8502
Testing bandwidth: 3.366666666666667
Bandwidth: 3.366666666666667, Mean CV Accuracy: 0.8502
Testing bandwidth: 3.911111111111112
Bandwidth: 3.911111111111112, Mean CV Accuracy: 0.8502
Testing bandwidth: 4.455555555555556
Bandwidth: 4.455555555555556, Mean CV Accuracy: 0.8509
Testing bandwidth: 5.0
Bandwidth: 5.0, Mean CV Accuracy: 0.8509

✅ Best bandwidth: 4.455555555555556 (CV Accuracy: 0.8509)

🎯 Final Test Accuracy with best bandwidth (4.455555555555556): 0.8658


In [30]:
import joblib

# Save the trained model
joblib.dump(model, 'kde_naive_bayes_model.pkl')

['kde_naive_bayes_model.pkl']

In [33]:
overview_features = [col for col in X_train.columns if 'overview' in col and 'missing' not in col]
joblib.dump(overview_features, "kde_overview_feature_names.pkl")

['kde_overview_feature_names.pkl']